# Explore the journeys behind your metrics

Start with an interactive path graph, then follow the question that matters to you: compare groups, open up a funnel, inspect an error, find a pattern, discover behavior types or zoom out to repeat visits.

This notebook accompanies the [README](https://github.com/retentioneering/retentioneering-tools). It uses Retentioneering's bundled **synthetic e-commerce dataset**; you do not need to upload any data. Run the setup and first graph, then try any of the numbered questions. Use Python 3.10–3.13.

[Installation help](https://retentioneering.com/docs/installation) · [Full interactive tour](https://colab.research.google.com/github/retentioneering/retentioneering-tools/blob/master/notebooks/retentioneering_5_tour.ipynb)


In [ ]:
%pip install -q retentioneering

In [ ]:
import os

os.environ["RETENTIONEERING_NO_TRACK"] = "1"

import retentioneering as rete

stream = rete.datasets.load_ecom()
print("Retentioneering", rete.__version__)
print(
    "Events:",
    len(stream.df),
    "| Users:",
    stream.df["user_id"].nunique(),
    "| Sessions:",
    stream.df["session_id"].nunique(),
)

## Your first graph

**Click an event to focus on its connections.** The settings panel lets you change the path level, edge metric and comparison groups. Start with sessions; switching to `path_col="user_id"` follows a user's history across visits. For the saved highlighted purchase route shown in the README GIF, open the full tour linked above.


In [ ]:
stream.transition_graph(path_col="session_id")

## 1. Conversion dropped. Where did the paths change?

Compare activity during the affected period with the rest of the log:

In [ ]:
periods = stream.add_segment("period", time_range=("2024-05-19", "2024-06-07"))
periods.transition_graph(path_col="session_id", diff=("period", "inside", "outside"))

Click `payment_details` to inspect the routes toward `purchase`, `payment_error` and `support_chat`. With this comparison order, red means a higher next-step probability inside the window; blue means lower. This locates a behavioral difference to investigate.

The same approach works for mobile versus desktop, acquisition channels, or experiment groups. Use comparable observation windows and check the counts behind a difference. A path comparison supports diagnosis; a causal or A/B-test conclusion needs the corresponding study design.

[Interactive graph demo](https://retentioneering.com/docs/widgets/transition-graph) · [Comparison recipes](https://retentioneering.com/docs/recipes)

## 2. Where do users leave the funnel—and what do they do instead?

Count sessions that complete the steps in order:

In [ ]:
steps = ["cart", "shipping_details", "purchase"]
stream.funnel(steps=steps, path_col="session_id")

Then compare sessions that stopped at the shipping stage of this funnel with those that completed it:

In [ ]:
stages = stream.add_segment("stage", funnel_events=steps, path_col="session_id")
stages.step_matrix(
    anchor="shipping_details",
    max_steps=6,
    path_col="session_id",
    diff=("stage", "shipping_details", "purchase"),
)

The matrix aligns sessions on the shipping step, so you can inspect the surrounding actions in each group. Try the same workflow for signup → activation → first payment.



[Interactive matrix demo](https://retentioneering.com/docs/widgets/step-matrix) · [Static view](.github/readme/step-matrix.png) · [Funnel definitions](https://retentioneering.com/docs/widgets/funnel)

## 3. What happens around an error or another key event?

Show the two steps on either side of a payment error:

In [ ]:
stream.step_sankey(
    path_pattern="payment_error",
    step_window=2,
    path_col="session_id",
)

Follow the branches around the event: where did sessions come from, and did they move toward retrying, support or the end of the session? Change the anchor to an event that matters in your product.

[Interactive Sankey demo](https://retentioneering.com/docs/widgets/step-sankey) · [Anchoring paths](https://retentioneering.com/docs/path-patterns)

## 4. Which sessions match a behavior I care about?

Find sessions that opened the cart, then ended without reaching shipping or support:

In [ ]:
abandoned = stream.filter_paths(
    {
        "metric": "matches_pattern",
        "op": "=",
        "value": True,
        "metric_args": {"pattern": "cart->[^shipping_details|support_chat]*->path_end"},
    },
    path_col="session_id",
)
abandoned.step_sankey(path_col="session_id", max_steps=6)

Patterns describe sequences with event names. Use them to select complete paths, measure a behavior or cut a journey around a meaningful sequence.

[Interactive pattern examples](https://retentioneering.com/docs/path-patterns) · [Path filtering](https://retentioneering.com/docs/data-processors/filter-paths)

## 5. Do users have different ways of using the product?

Group sessions by how often each event occurs:

In [ ]:
stream.cluster_analysis(
    features=[{"metric": "event_count_bulk"}],
    method_args={"n_clusters": "3-5"},
    path_col="session_id",
)

Inspect the event heatmap, switch between candidate cluster counts and explore each group's paths. In the live notebook, adjust features in the sidebar and use **Save clusters** to copy the generated code or apply the labels to the stream.



Treat clusters as exploratory descriptions; check stability and usefulness before using them for targeting.

[Interactive cluster demo](https://retentioneering.com/docs/widgets/cluster-analysis) · [Static view](.github/readme/cluster-analysis.png) · [Behavioral features](https://retentioneering.com/docs/path-metrics)

## 6. How does behavior change across visits or long pauses?

Zoom out: turn each session into one event, using the demo's supplied lifecycle label as its name. Now every Sankey step is a visit:

In [ ]:
visits = stream.collapse_events(group_col="session_id", name={"col": "user_lifecycle"})
visits.step_sankey(path_col="user_id", max_steps=8)

For your own data, you can name sessions by what happened inside them—for example, browsing, buying or contacting support. [Event aggregation](https://retentioneering.com/docs/data-processors/collapse-events) lets the same analysis move from individual clicks to sequences of visits.

You can also mark observed inactivity gaps using a native processor:

In [ ]:
pauses = stream.add_events("inactive_30d", churn={"inactivity_days": 30})
pauses.step_matrix(path_pattern="inactive_30d", step_window=2)

Here the marker follows the last activity before an observed gap longer than 30 days. A user may return later; this is an inactivity definition, not proof of permanent churn. Choose a threshold and observation window that suit your product.

[Interactive lifecycle examples](https://retentioneering.com/docs/data-processors/to-daily-states) · [Inactivity markers](https://retentioneering.com/docs/data-processors/add-events)

## Check and share

Inspect the table behind a graph, then export the interactive view. The HTML can be opened without Python; recalculation needs the notebook.


In [ ]:
table = stream.transition_graph_data(path_col="session_id", edge_weight="count")
graph = stream.transition_graph(path_col="session_id")
graph.export_html("session-journeys.html")
table.head()

## Try your own events

The default columns are `user_id`, `event`, and `timestamp`. For other columns, use a [schema](https://retentioneering.com/docs/eventstream#schema). Your data can also describe agent runs, tickets or other ordered cases.

```python
import pandas as pd
events = pd.read_csv("events.csv", parse_dates=["timestamp"])
own_stream = rete.Eventstream(events)
own_stream.transition_graph()
```

Before interpreting your own results, inspect identities, duplicates, event order and the observation window. The bundled teaching data itself contains duplicate rows and timestamp ties; its planted patterns demonstrate the API, not causal evidence about a real business. See [path analysis](https://retentioneering.com/docs/path-analysis) for interpretation and [the dataset discussion](https://github.com/retentioneering/retentioneering-tools/issues/88) for its ordering limitations.

Next: [analyze agent runs](agent_path_analysis.ipynb) · [more recipes](https://retentioneering.com/docs/recipes) · [ask a question](https://github.com/retentioneering/retentioneering-tools/issues)
